# Syllable-error scoring

Raw and normalized scores from aligned Apple Siri transcriptions of 74 participants.

## Step 1: Load the aligned transcriptions

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
df = pd.read_csv(ROOT / 'syllableseparation_apple_corrected.csv', dtype={'sub_id': str})
assert len(df) == 74 and df['sub_id'].is_unique
assert df['disease'].value_counts().to_dict() == {0: 37, 1: 37}
display(df[['sub_id', 'disease']].head())

## Step 2: Define the position-specific accepted transcriptions

In [ ]:
accepted = {
    1: [['ปู'], ['ม้า'], ['วิ่ง'], ['ไป'], ['มา'], ['บน'], ['ใบ'], ['ไม้']],
    2: [['ชาว'], ['ไร่', 'ไล่'], ['ตัด'], ['ต้น'], ['สน'], ['ทำ', 'ธรรม'], ['ท่อน'], ['ซุง']],
    3: [['อี', 'อีก'], ['กา'], ['คอย', 'คอยน์'], ['คาบ'], ['งู'], ['คาบ'], ['ไก่']],
    4: [['เพียง'], ['แค่'], ['ฝน'], ['ตก'], ['ลง'], ['ที่'], ['หน้า'], ['ต่าง'], ['ใน'], ['บาง'], ['ครา', 'คา']]
}
sentence_lengths = [8, 8, 7, 11]

## Step 3: Calculate and save the scores

In [ ]:
result = df[['sub_id', 'disease']].copy()
for sentence, alternatives in accepted.items():
    score = np.zeros(len(df), dtype=int)
    for position, allowed in enumerate(alternatives, 1):
        score += (~df[f's{sentence}_{position}'].isin(allowed)).to_numpy(dtype=int)
    extras = [column for column in df if column.startswith(f'e{sentence}_')]
    for column in extras:
        score += df[column].astype(str).ne('0').to_numpy(dtype=int)
    result[f'error_{sentence}'] = score
for sentence, length in enumerate(sentence_lengths, 1):
    result[f'error_rate_{sentence}'] = result[f'error_{sentence}'] / length
result.to_csv(ROOT / 'Syllableerror_scores_raw_and_normalized.csv', index=False)
display(result.head())